# [Chapter 17 — Integrated Case Studies: Introduction and COVID-19, §17.6] NPI Impact: Reconstructing $\mathcal{R}_e(t)$ from Incidence

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 17 — Integrated Case Studies: Introduction and COVID-19
**Considerations developed:** 12 (basic vs effective $R$ misapplication), 13 (equilibrium vs invasion conflation)
**Estimated runtime:** ≤ 20 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_17_covid/02_NPI_impact_via_Re_t.ipynb)


## What this notebook does

Reconstructs the time-varying effective reproductive number $\mathcal{R}_e(t)$ from the synthetic COVID incidence data using the Cori et al. (2013) sliding-window estimator. Demonstrates how a well-implemented intervention manifests as a downward step in $\mathcal{R}_e(t)$, and how reporting $\mathcal{R}_0$ alone (a single number) misses this dynamic — the central Consideration~12 operational point.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from scipy.optimize import minimize
import csv, json
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_17
from shared.verification import assert_within_tolerance
set_seed_chapter_17()
book_style()

# Path to the synthetic-data root for the case studies
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'data'))


## Load data and synthetic-intervention overlay

We add a synthetic intervention at day 30 that reduces effective contact rate by 50% — this would correspond to a school-and-workplace closure in a real outbreak. Then we re-simulate and re-export the post-intervention incidence to demonstrate the $\mathcal{R}_e(t)$ reconstruction.

In [ ]:
from shared.parameters import baseline_chapter_17
p = baseline_chapter_17()
N = p['N']

def seir_with_npi(t_npi, reduction):
    def rhs(y, t):
        S, E, I, R = y
        cI_eff = p['c_I'] * (1 - reduction if t > t_npi else 1.0)
        inc = cI_eff * p['beta'] * S * I / N
        return [-inc, inc - E/p['tau_E'], E/p['tau_E'] - I/p['tau_R'], I/p['tau_R']]
    t = np.linspace(0, 90, 91)
    y0 = [N - 5000 - 5000, 5000.0, 5000.0, 0.0]
    sol = odeint(rhs, y0, t)
    new_inf = -np.diff(np.concatenate([[y0[0]], sol[:, 0]]))
    return t, new_inf

t_no_npi, inf_no_npi = seir_with_npi(t_npi=999, reduction=0)
t_with_npi, inf_with_npi = seir_with_npi(t_npi=30, reduction=0.5)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_no_npi, inf_no_npi, color=BOOK_COLORS['primary'], lw=2, label='no NPI')
ax.plot(t_with_npi, inf_with_npi, color=BOOK_COLORS['secondary'], lw=2, label='NPI on day 30')
ax.axvline(30, ls=':', color='gray', alpha=0.6)
ax.set_xlabel('day')
ax.set_ylabel('new infections (true)')
ax.set_title('Counterfactual: with vs without NPI')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## $\mathcal{R}_e(t)$ reconstruction

Using the Cori-method approximation (window-averaged ratio of new infections to weighted sum of recent infections):
$$\hat{\mathcal{R}}_e(t) = \frac{I(t)}{\sum_s w_s I(t-s)}$$
where $w_s$ is the assumed generation-time distribution (here: exponential with mean $\tau_R$).

In [ ]:
def reconstruct_Re(incidence, tau_R, window=7):
    n = len(incidence)
    w = np.exp(-np.arange(1, 21)/tau_R)
    w = w / w.sum()
    Re = np.full(n, np.nan)
    for t in range(20, n):
        denom = float(np.dot(w, incidence[t-20:t][::-1]))
        if denom > 0.5:
            Re[t] = incidence[t] / denom
    # Smooth with sliding mean
    smooth = np.full(n, np.nan)
    for t in range(window, n):
        vals = Re[t-window:t]
        if not np.all(np.isnan(vals)):
            smooth[t] = np.nanmean(vals)
    return smooth

Re_no_npi = reconstruct_Re(inf_no_npi, tau_R=p['tau_R'])
Re_with_npi = reconstruct_Re(inf_with_npi, tau_R=p['tau_R'])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(t_no_npi, Re_no_npi, color=BOOK_COLORS['primary'], lw=2, label='no NPI')
ax.plot(t_with_npi, Re_with_npi, color=BOOK_COLORS['secondary'], lw=2, label='NPI day 30')
ax.axvline(30, ls=':', color='gray', alpha=0.6, label='NPI start')
ax.axhline(1.0, ls='--', color='black', alpha=0.4)
ax.set_xlabel('day')
ax.set_ylabel(r'$\hat{\mathcal{R}}_e(t)$')
ax.set_title('$\\mathcal{R}_e(t)$ reconstruction reveals what $\\mathcal{R}_0$ alone hides')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 4)
fig.tight_layout()
plt.show()

# Verify: Re should drop visibly after intervention
Re_pre = np.nanmean(Re_with_npi[25:30])
Re_post = np.nanmean(Re_with_npi[45:60])
print(f'Mean Re pre-NPI:  {Re_pre:.2f}')
print(f'Mean Re post-NPI: {Re_post:.2f}')
print(f'Reduction: {(Re_pre - Re_post)/Re_pre*100:.0f}%')
assert Re_post < Re_pre, 'NPI should reduce Re'
print('\nVerified: Re reconstruction reveals the intervention effect.')


## What's next

- Notebook 03 addresses under-reporting correction in the fitting workflow.
- The Cori method is approximate; rigorous treatments use the empirical generation-time distribution (Consideration 10).
- Real interventions are not step functions; smooth ramps and partial compliance change the $\mathcal{R}_e(t)$ trajectory shape — a useful exercise extension.